# Compare two datasets

Finds differences between a **source** dataset and a **compare** dataset (for example the same
output before and after a code change): schema, row counts, then row-level and per-column differences
on key columns. Differing records are written to CSV in S3.

**Before running:** both datasets exist in S3 and the parameters below are set. Data is read lazily;
the only thing written is the diff CSVs at `OUTPUT_PATH`.

**Stop the notebook instance when you finish** — SageMaker bills while it is running.

## Parameters

- `SOURCE_PATH`, `COMPARE_PATH` — the two datasets to compare (S3 folders of parquet files).
- `STORAGE_OPTIONS` — Polars S3 options; leave `None` unless the region isn't found.
- `OUTPUT_PATH` — S3 folder the diff CSVs are written to.
- `KEY_COLUMNS` — columns that identify a record; rows are matched on these. Keys must be unique
  within each dataset, otherwise the per-column comparison fans out.
- `SLICE_COLUMN` — one of the key columns; each of its values is compared separately to keep memory down.
- `COMPARE_COLUMNS` — columns checked for differences (`None` = all source columns).

In [ ]:
SOURCE_PATH = "s3://<bucket>/domain=<domain>/dataset=<source_dataset>/version=<version>/"
COMPARE_PATH = "s3://<bucket>/domain=<domain>/dataset=<compare_dataset>/version=<version>/"
OUTPUT_PATH = "s3://<bucket>/domain=<domain>/dataset=compare_outputs/"
STORAGE_OPTIONS = None  # e.g. {"aws_region": "eu-west-2"} if Polars can't find the region

# Column names are defined once, here. Edit them for your datasets.
KEY_COLUMNS = ["location_id", "year"]
SLICE_COLUMN = "year"  # None = compare in one pass
COMPARE_COLUMNS = None  # None = every source column; or a list of column names

## Setup

In [ ]:
import polars as pl

## Load both datasets

Scans are lazy: nothing is read until a result is collected.

In [ ]:
source_lf = pl.scan_parquet(SOURCE_PATH, storage_options=STORAGE_OPTIONS)
compare_lf = pl.scan_parquet(COMPARE_PATH, storage_options=STORAGE_OPTIONS)

## Schema check

Columns found in only one dataset, and columns whose type differs (for example `Float32` vs
`Float64`). Columns in only one dataset are left out of the comparison below.

In [ ]:
source_schema = source_lf.collect_schema()
compare_schema = compare_lf.collect_schema()

print("Only in source: ", sorted(set(source_schema) - set(compare_schema)))
print("Only in compare:", sorted(set(compare_schema) - set(source_schema)))
print("Type differs:   ", {
    column: (str(source_schema[column]), str(compare_schema[column]))
    for column in source_schema.keys() & compare_schema.keys()
    if source_schema[column] != compare_schema[column]
})

## Row counts and duplicate keys

Duplicate keys (a `KEY_COLUMNS` combination appearing more than once in a dataset) should be 0: they
make the per-column comparison fan out and inflate the counts.

In [ ]:
def count_rows(lf: pl.LazyFrame) -> int:
    return lf.select(pl.len()).collect().item()


def count_duplicate_keys(lf: pl.LazyFrame) -> int:
    return lf.group_by(KEY_COLUMNS).len().filter(pl.col("len") > 1).select(pl.len()).collect().item()


source_rows, compare_rows = count_rows(source_lf), count_rows(compare_lf)
print(f"rows            source: {source_rows:,}   compare: {compare_rows:,}   "
      f"{'match' if source_rows == compare_rows else 'DIFFER'}")
print(f"duplicate keys  source: {count_duplicate_keys(source_lf):,}   compare: {count_duplicate_keys(compare_lf):,}")

## Prepare the comparison

`prepare` keeps only the compared columns and trims strings, treating `""` and `" "` as null. Columns
whose type differs between the datasets are cast to a common type; matching columns are left alone. The
common type is the source's, except where both datasets hold floats: those are compared as `Float32`, so a
precision change alone (`Float64` vs `Float32`) is not a value difference.

In [ ]:
absent = [c for c in KEY_COLUMNS + (COMPARE_COLUMNS or []) if c not in source_schema]
if absent:
    raise ValueError(f"Not in the source dataset: {absent}")
absent = [c for c in KEY_COLUMNS if c not in compare_schema]
if absent:
    raise ValueError(f"Key columns missing from the compare dataset: {absent}")

compare_columns = list(dict.fromkeys(
    KEY_COLUMNS
    + [c for c in (COMPARE_COLUMNS or source_schema.names()) if c in compare_schema]
))
value_columns = [c for c in compare_columns if c not in KEY_COLUMNS]
if not value_columns:
    raise ValueError("No columns to compare besides KEY_COLUMNS")
string_columns = [c for c in compare_columns if source_schema[c] == pl.String]
common_types = {
    c: pl.Float32 if source_schema[c].is_float() and compare_schema[c].is_float() else source_schema[c]
    for c in compare_columns
    if source_schema[c] != compare_schema[c]
}


def prepare(lf: pl.LazyFrame, schema: pl.Schema) -> pl.LazyFrame:
    """Select the compared columns, cast those whose type differs, blank empty strings to null."""
    return lf.select(compare_columns).with_columns(
        [pl.col(c).cast(t) for c, t in common_types.items() if schema[c] != t]
    ).with_columns(
        pl.when(pl.col(c).str.strip_chars() == "")
        .then(None)
        .otherwise(pl.col(c).str.strip_chars())
        .alias(c)
        for c in string_columns
    )

## Find the differences

`write_diffs` writes two CSVs per slice to `OUTPUT_PATH`, overwriting any from an earlier run; a file
with only a header means no differences:

- `row_diff_<slice>.csv` — rows whose compared columns are not identical, tagged `source_only` or `compare_only`.
- `column_diff_<slice>.csv` — for keys present in both datasets, one row per differing column with the
  source and compare values.

Nulls compare equal to nulls.

In [ ]:
sink_options = {"storage_options": STORAGE_OPTIONS} if STORAGE_OPTIONS else {}


def write_diffs(source: pl.LazyFrame, compare: pl.LazyFrame, label: str) -> None:
    """Write row-level and per-column differences for one slice to CSV."""
    source, compare = prepare(source, source_schema), prepare(compare, compare_schema)

    row_diff = pl.concat([
        source.join(compare, on=compare_columns, how="anti", nulls_equal=True)
        .with_columns(pl.lit("source_only").alias("found_in")),
        compare.join(source, on=compare_columns, how="anti", nulls_equal=True)
        .with_columns(pl.lit("compare_only").alias("found_in")),
    ])

    matched = source.join(compare, on=KEY_COLUMNS, how="inner", suffix="_compare", nulls_equal=True)
    column_diff = pl.concat([
        matched.filter(pl.col(c).ne_missing(pl.col(f"{c}_compare"))).select(
            *KEY_COLUMNS,
            pl.lit(c).alias("column"),
            pl.col(c).cast(pl.String).alias("source_value"),
            pl.col(f"{c}_compare").cast(pl.String).alias("compare_value"),
        )
        for c in value_columns
    ])

    for name, diff in [("row_diff", row_diff), ("column_diff", column_diff)]:
        path = f"{OUTPUT_PATH}{name}_{label}.csv"
        diff.sink_csv(path, **sink_options)
        count = pl.scan_csv(path, storage_options=STORAGE_OPTIONS).select(pl.len()).collect().item()
        print(f"{label} {name}: {count:,} differing records")

## Run the comparison

Runs `write_diffs` once per `SLICE_COLUMN` value found in either dataset, including null (or once if
`SLICE_COLUMN` is `None`).

In [ ]:
if SLICE_COLUMN:
    slice_values = (
        pl.concat([source_lf.select(SLICE_COLUMN), compare_lf.select(SLICE_COLUMN)])
        .unique()
        .collect()
        .to_series()
        .sort()
        .to_list()
    )
    slices = [
        ("null", pl.col(SLICE_COLUMN).is_null()) if value is None
        else (str(value), pl.col(SLICE_COLUMN) == value)
        for value in slice_values
    ]
else:
    slices = [("all", pl.lit(True))]

for label, keep in slices:
    write_diffs(source_lf.filter(keep), compare_lf.filter(keep), label)